# LTX-2.5 22B on Kaggle — Wan2GP + Distilled NVFP4

Designed for **Kaggle T4**. This notebook keeps the large LTX-2.5 model and text encoder in `/tmp` to avoid filling `/kaggle/working`.

**Target test:** LTX-2.5 22B Distilled NVFP4, 480p, ~5 seconds, **8 diffusion steps + 3 extra post-processing/upscaling steps**.

The notebook's existing code cells are already configured for the LTX-2.5 checkpoint and Gemma 4 text encoder.

## 1. Check GPU and disk

In [ ]:
!nvidia-smi

import torch, os
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), 
          f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")

## 2. Clone Wan2GP

In [ ]:
%cd /kaggle/working
import os
if not os.path.exists('/kaggle/working/Wan2GP'):
    !git clone https://github.com/deepbeepmeep/Wan2GP.git
%cd /kaggle/working/Wan2GP

## 3. Install Wan2GP dependencies

In [ ]:
%cd /kaggle/working/Wan2GP

!pip install -r requirements.txt

## 4. Create temporary LTX-2.5 model directory

In [ ]:
!mkdir -p /tmp/ltx2.5
!df -h

## 5. Download the LTX-2.5 22B Distilled NVFP4 checkpoint

The checkpoint is about **14 GB**, so it is stored in `/tmp/ltx2.5` instead of `/kaggle/working`.

In [ ]:
from huggingface_hub import hf_hub_download
import os

base = hf_hub_download(
    repo_id='DeepBeepMeep/LTX-2',
    filename='ltx-2.5-22b-distilled_diffusion_model_nvfp4.safetensors',
    local_dir='/tmp/ltx2.5'
)

print('LTX-2.5:', base)
!ls -lh /tmp/ltx2.5

## 6. Download the Gemma 4 12B INT8 ConvRot text encoder

This is the LTX-2.5 text encoder used by the current Wan2GP LTX-2.5 configuration.
```

In [ ]:
encoder = hf_hub_download(
    repo_id='DeepBeepMeep/LTX-2',
    filename='gemma4-12b-ltx-v1/gemma4-12b-ltx-v1_int8_convrot.safetensors',
    local_dir='/tmp/ltx2.5'
)

print('Encoder:', encoder)
!du -sh /tmp/ltx2.5

## 7. Launch Wan2GP

**Run this cell last.** It keeps the notebook cell occupied while Wan2GP is running.

Open the Gradio URL printed by the cell.

After opening Wan2GP, select the **LTX-2.5 22B Distilled NVFP4** model/workflow and use the local checkpoints downloaded above.

In [ ]:
%cd /kaggle/working/Wan2GP

!python wgp.py --share

## 8. LTX-2.5 Checkpoint Paths

### Main Checkpoint

```text
/tmp/ltx2.5/ltx-2.5-22b-distilled_diffusion_model_nvfp4.safetensors
```

### Text Encoder Checkpoint

```text
/tmp/ltx2.5/gemma4-12b-ltx-v1/gemma4-12b-ltx-v1_int8_convrot.safetensors
```

### LTX-2.5 supporting files

Wan2GP will handle the required LTX-2.5 supporting components, including:

- Video VAE
- Diffusion Video VAE
- Audio VAE
- Vocoder
- Text embedding projection
- Video/audio embedding connectors
- Spatial and temporal upscalers

On Kaggle, these files can be downloaded automatically by Wan2GP when required.

## 9. Suggested First-Generation Settings

For the first LTX-2.5 test:

* Model: **LTX-2.5 22B Distilled NVFP4**
* Steps: **8**
* Resolution: **480p**
* Duration: **~5 seconds**
* Batch size: **1**
* Keep other acceleration and quality options at their defaults for the first test.

### Expected Generation Time on Kaggle T4

With this setup, generation can be quite slow because the T4 does not have optimized NVFP4 kernels and therefore uses the fallback implementation.

As a rough estimate:

* **5-second video:** around **15 minutes**
* **10-second video:** around **30 minutes**

Actual generation time may vary depending on resolution, GPU availability, settings, and additional processing stages.

### What to Expect on a Kaggle T4

The NVFP4 checkpoint can run, but the T4 may show:

NVFP4: kernels unavailable; using fallback.
NVFP4: linear fallback

This is expected when optimized NVFP4 kernels are unavailable. The model can still generate successfully, but the fallback makes generation considerably slower.

The **8-step diffusion pass** is the main generation. Wan2GP may also perform separate decoding, upscaling, or post-processing afterward, which can show additional progress bars such as **3/3**.

**Do not interpret those additional stages as extra diffusion steps.**

## 10. GPU monitoring (run in a separate cell after starting generation)

Run this while a generation is in progress to monitor GPU memory and utilization.

In [ ]:
!nvidia-smi